# 🌌 Courbes de lumière `TAG` — Fink/LSST *(Deep Drilling Fields)*

Pour chaque objet **dans les Deep Drilling Fields (ECDFS / EDFS)**, ce notebook génère une **figure de courbe de lumière** avec :
- **Panneau du haut** : courbe de lumière en **flux PSF (nJy)** — toutes les bandes `ugrizy` superposées
- **Panneau du bas** : courbe de lumière en **magnitude AB** — toutes les bandes `ugrizy` superposées

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources`

> **Option DDF activée** — les alertes sont pré-filtrées spatialement sur les champs ECDFS et EDFS avant le tracé.

In [ ]:
# ── Liste des tags disponibles ────────────────────────────────────────────────
import requests

r = requests.get("https://api.lsst.fink-portal.org/api/v1/tags")
tags = r.json()
for tag, doc in tags.items():
    print(f"{tag:40s}  {doc}")

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_ALERTS   = 100                       # Nombre d'alertes à récupérer
STARTDATE  = None # "2025-10-01 00:00:00"                     # "2026-01-01 00:00:00" ou None
STOPDATE   = None # "2026-03-01 00:00:00"                    # "2026-03-01 00:00:00" ou None
BASE_URL   = "https://api.lsst.fink-portal.org"
TAG        = "extragalactic_lt20mag_candidate"     # Choisir parmi la liste ci-dessus
SAVE_FIGURES  = False                  # Sauvegarde PDF + PNG de chaque figure

# ── Option DDF : voir cellule '1b. Deep Drilling Fields' ──────────────
# Mettre DDF_FILTER = True/False dans la cellule dédiée ci-dessous.
OUTPUT_DIR    = "lightcurves"         # Dossier de sortie
# ─────────────────────────────────────────────────────────────────────────────

## 1b. Deep Drilling Fields — définition et filtrage

Option activée : seules les alertes tombant dans un des **DDFs** ci-dessous
sont conservées pour le tracé des courbes de lumière.

| Champ | RA (°) | Dec (°) | Rayon (°) | Description |
|-------|--------|---------|-----------|-------------|
| ECDFS | 53.16  | −28.10  | 1.0       | Extended Chandra Deep Field South |
| EDFS  | 59.10  | −48.73  | 1.0       | Euclid Deep Field South           |

> Références DP1 :
> [ECDFS tutorial 301.4](https://dp1.lsst.io/tutorials/notebook/301/notebook-301-4.html) ·
> [EDFS tutorial 301.5](https://dp1.lsst.io/tutorials/notebook/301/notebook-301-5.html)


In [ ]:
# ─── Option Deep Drilling Fields ─────────────────────────────────────────────
DDF_FILTER = True          # True  → ne garder que les alertes dans les DDFs
                            # False → comportement identique au notebook original

# Définition des champs — coordonnées tirées des tutoriels DP1 (301.4 / 301.5)
DDF_FIELDS = {
    "ECDFS": {"ra": 53.16, "dec": -28.10, "radius_deg": 0.5,
              "color": "#1E90FF",
              "description": "Extended Chandra Deep Field South — champ principal DP1"},
    "EDFS":  {"ra": 59.10, "dec": -48.73, "radius_deg": 1.0,
              "color": "#FF8C00",
              "description": "Euclid Deep Field South"},
}
# Pour ne sélectionner qu'un seul champ, commenter l'autre ligne, par exemple :
DDF_FIELDS = {"ECDFS": DDF_FIELDS["ECDFS"]}
# DDF_FIELDS = {"EDFS": DDF_FIELDS["EDFS"]}
# ─────────────────────────────────────────────────────────────────────────────


## 2. Imports

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
from IPython.display import display as ipy_display, HTML

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')

def mjd_to_datetime(mjd_series):
    """Convertit une Series MJD en Series de Timestamps pandas (UTC)."""
    return MJD_EPOCH + pd.to_timedelta(pd.to_numeric(mjd_series, errors='coerce'), unit='D')

def mjd_to_datestr(mjd):
    """Convertit un scalaire MJD en chaîne 'YYYY-MM-DD'."""
    try:
        return (MJD_EPOCH + pd.to_timedelta(float(mjd), unit='D')).strftime('%Y-%m-%d')
    except Exception:
        return str(mjd)

if SAVE_FIGURES:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Imports OK ✓")

## 3. Style matplotlib

In [ ]:
mpl.rcParams.update({
    'font.size'         : 11,
    'axes.titlesize'    : 16,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 16,
    'xtick.labelsize'   : 16,
    'ytick.labelsize'   : 16,
    'legend.fontsize'   : 16,
    'figure.titlesize'  : 13,
    'figure.titleweight': 'bold',
})


# Couleurs LSST standard par bande
FILTER_COLORS = {
    'u': '#7B2FBE',   # violet
    'g': '#0077BB',   # bleu
    'r': '#33AA77',   # vert
    'i': '#DDAA33',   # jaune-or
    'z': '#BB5500',   # orange
    'y': '#AA0000',   # rouge sombre
}



FILTER_ORDER = ['u', 'g', 'r', 'i', 'z', 'y']   # ordre naturel LSST

ZP_AB = 2.5 * np.log10(3631e9)   # ≈ 31.4 pour F en nJy

print("Style OK ✓")

## 4. Fonctions API

In [ ]:
def fetch_tag_alerts(tag, n, startdate=None, stopdate=None):
    """Récupère les N dernières alertes pour un tag donné."""
    payload = {"tag": tag, "n": n, "output-format": "json"}
    if startdate:
        payload["startdate"] = startdate
    if stopdate:
        payload["stopdate"] = stopdate
    resp = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        raise ValueError(f"Aucune alerte pour le tag '{tag}'.")
    return pd.DataFrame(data)


def fetch_lightcurve(dia_object_id):
    """
    Télécharge la courbe de lumière complète d'un diaObjectId.
    Retourne un DataFrame avec colonnes : midpointMjdTai, psfFlux,
    psfFluxErr, band, diaSourceId.
    """
    payload = {
        "diaObjectId"  : str(dia_object_id),
        "columns"      : "r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,r:diaSourceId",
        "output-format": "json",
    }
    resp = requests.post(f"{BASE_URL}/api/v1/sources", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '') for c in df.columns]
    for col in ['midpointMjdTai', 'psfFlux', 'psfFluxErr']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


print("Fonctions API OK ✓")

## 4b. Fonction de filtrage spatial DDF

Utilise `astropy.SkyCoord.separation()` pour calculer la séparation angulaire
entre chaque alerte et le centre de chaque DDF, puis ne conserve que les alertes
dont la séparation est inférieure au rayon du champ.


In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

def filter_alerts_in_ddfs(df_tags, ddf_fields, ra_col=None, dec_col=None):
    """
    Filtre les alertes pour ne conserver que celles dans les DDFs.

    Paramètres
    ----------
    df_tags   : DataFrame retourné par fetch_tag_alerts()
    ddf_fields: dict  DDF_FIELDS  (nom → ra/dec/radius_deg/color/description)
    ra_col    : str | None  — colonne RA  (auto-détectée si None)
    dec_col   : str | None  — colonne Dec (auto-détectée si None)

    Retourne
    --------
    DataFrame filtré avec deux colonnes supplémentaires :
      'ddf_name'    — nom du DDF le plus proche
      'ddf_sep_deg' — séparation angulaire en degrés
    """
    import numpy as np

    if df_tags.empty:
        return df_tags

    # ── Auto-détection des colonnes RA/Dec ────────────────────────────────
    def _find_col(df, hints):
        for h in hints:
            matches = [c for c in df.columns if h in c.lower()]
            if matches:
                return matches[0]
        return None

    if ra_col is None:
        ra_col  = _find_col(df_tags, ["r:ra", "ra"])
    if dec_col is None:
        dec_col = _find_col(df_tags, ["r:dec", "dec"])

    if ra_col is None or dec_col is None:
        raise KeyError(
            f"Impossible de trouver les colonnes RA/Dec. "
            f"Colonnes disponibles : {list(df_tags.columns)}"
        )

    alerts_coord = SkyCoord(
        ra  = df_tags[ra_col].astype(float).values  * u.deg,
        dec = df_tags[dec_col].astype(float).values * u.deg,
    )

    matched_mask = np.zeros(len(df_tags), dtype=bool)
    matched_name = np.full(len(df_tags), "", dtype=object)
    matched_sep  = np.full(len(df_tags), np.nan)

    for ddf_name, ddf in ddf_fields.items():
        center  = SkyCoord(ra=ddf["ra"] * u.deg, dec=ddf["dec"] * u.deg)
        seps    = alerts_coord.separation(center).deg
        in_ddf  = seps <= ddf["radius_deg"]
        new_match = in_ddf & ~matched_mask        # évite les doublons
        matched_mask[new_match] = True
        matched_name[new_match] = ddf_name
        matched_sep[new_match]  = seps[new_match]
        print(f"  {ddf_name:6s}: {in_ddf.sum():4d} alertes dans r < {ddf['radius_deg']}°")

    df_out = df_tags[matched_mask].copy()
    df_out["ddf_name"]    = matched_name[matched_mask]
    df_out["ddf_sep_deg"] = matched_sep[matched_mask]
    df_out = df_out.sort_values(["ddf_name", "ddf_sep_deg"]).reset_index(drop=True)
    return df_out

def plot_ddf_map(df_alerts, ddf_fields, ra_col=None, dec_col=None):
    """Mini-carte RA/Dec montrant les alertes retenues dans chaque DDF."""
    import numpy as np
    from matplotlib.patches import Circle

    def _find_col(df, hints):
        for h in hints:
            matches = [c for c in df.columns if h in c.lower()]
            if matches:
                return matches[0]
        return None

    if ra_col is None:
        ra_col  = _find_col(df_alerts, ["r:ra",  "ra"])
    if dec_col is None:
        dec_col = _find_col(df_alerts, ["r:dec", "dec"])

    n_ddf = len(ddf_fields)
    fig, axes = plt.subplots(1, n_ddf, figsize=(5.5 * n_ddf, 5))
    if n_ddf == 1:
        axes = [axes]

    for ax, (ddf_name, ddf) in zip(axes, ddf_fields.items()):
        sub = df_alerts[df_alerts["ddf_name"] == ddf_name] if not df_alerts.empty else df_alerts
        circ = Circle((ddf["ra"], ddf["dec"]), ddf["radius_deg"],
                      fill=False, edgecolor=ddf["color"],
                      linewidth=1.8, linestyle="--", label=f"{ddf_name} footprint")
        ax.add_patch(circ)
        ax.plot(ddf["ra"], ddf["dec"], "+",
                color=ddf["color"], markersize=14, markeredgewidth=2)
        if ra_col and dec_col and not sub.empty:
            ax.scatter(sub[ra_col].astype(float), sub[dec_col].astype(float),
                       s=18, color=ddf["color"], alpha=0.65,
                       label=f"{len(sub)} alertes")
        ax.set_xlabel("RA (°)")
        ax.set_ylabel("Dec (°)")
        ax.set_title(f"{ddf_name}\n{ddf['description']}", fontsize=10)
        ax.invert_xaxis()
        ax.set_aspect("equal")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.25)

    fig.suptitle(f"Alertes FINK/LSST dans les DDFs — TAG : {TAG}", fontsize=11)
    plt.tight_layout()
    plt.show()

print("Fonctions DDF OK ✓")


## 5. Fonctions de tracé

In [ ]:
def flux_to_mag_ab(flux_njy, flux_err_njy):
    """Flux (nJy) → magnitude AB + erreur propagée. NaN si flux ≤ 0."""
    flux   = np.asarray(flux_njy,     dtype=float)
    flux_e = np.asarray(flux_err_njy, dtype=float)
    valid   = flux > 0
    mag     = np.full(flux.shape, np.nan)
    mag_err = np.full(flux.shape, np.nan)
    mag[valid]     = -2.5 * np.log10(flux[valid]) + ZP_AB
    mag_err[valid] = (2.5 / np.log(10)) * np.abs(flux_e[valid] / flux[valid])
    return mag, mag_err


def _prepare_lc(df):
    """Nettoyage commun : conversion types, drop NaN, ajout colonne date."""
    df = df.copy()
    for col in ['midpointMjdTai', 'psfFlux', 'psfFluxErr']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['midpointMjdTai', 'psfFlux', 'psfFluxErr'])
    df['date'] = mjd_to_datetime(df['midpointMjdTai'])
    return df


def _apply_date_axis(ax, show_xlabel=False):
    """Formattage commun de l'axe X (dates)."""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')
    ax.grid(True, alpha=0.2, linewidth=0.5, linestyle='--')
    if show_xlabel:
        ax.set_xlabel("Date (UTC)")


def _bands_in_order(lc):
    """Retourne les bandes présentes dans lc, dans l'ordre LSST ugrizy."""
    present = set(lc['band'].dropna().unique()) if 'band' in lc.columns else set()
    return [b for b in FILTER_ORDER if b in present]


def plot_lc_flux(ax, lc):
    """Panneau flux PSF (nJy) — toutes bandes superposées."""
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    if not required.issubset(lc.columns):
        ax.text(0.5, 0.5, "Données manquantes", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    df = _prepare_lc(lc)
    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return

    bands = _bands_in_order(df)
    for band in bands:
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')
        ax.errorbar(
            sub['date'], sub['psfFlux'], yerr=sub['psfFluxErr'],
            fmt='o', color=color, label=f"${band}$",
            markersize=5, linewidth=0.9,
            capsize=3, capthick=0.8, elinewidth=0.8, alpha=0.88)

    ax.axhline(0, color='gray', lw=0.7, ls=':', alpha=0.6)
    ax.set_ylabel("Flux PSF (nJy)")
    ax.legend(
        ncol=len(bands), loc='upper left',
        framealpha=0.85, handlelength=1.2,
        handletextpad=0.4, columnspacing=0.8)
    _apply_date_axis(ax, show_xlabel=False)


def plot_lc_mag(ax, lc):
    """Panneau magnitude AB — toutes bandes superposées."""
    required = {'midpointMjdTai', 'psfFlux', 'psfFluxErr', 'band'}
    if not required.issubset(lc.columns):
        ax.text(0.5, 0.5, "Données manquantes", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return
    df = _prepare_lc(lc)
    if df.empty:
        ax.text(0.5, 0.5, "Pas de données valides", ha='center', va='center',
                transform=ax.transAxes, color='gray')
        return

    mag, mag_err = flux_to_mag_ab(df['psfFlux'].values, df['psfFluxErr'].values)
    df = df.copy()
    df['mag']     = mag
    df['mag_err'] = mag_err

    bands = _bands_in_order(df)
    for band in bands:
        sub   = df[df['band'] == band].sort_values('date')
        color = FILTER_COLORS.get(band, '#888888')

        # Détections (flux > 0 → magnitude valide)
        det = sub.dropna(subset=['mag'])
        if not det.empty:
            ax.errorbar(
                det['date'], det['mag'], yerr=det['mag_err'],
                fmt='o', color=color, label=f"${band}$",
                markersize=5, linewidth=0.9,
                capsize=3, capthick=0.8, elinewidth=0.8, alpha=0.88)

        # Non-détections : flèches vers le bas (limite 3-sigma)
        nondet = sub[sub['mag'].isna()]
        if not nondet.empty:
            sigma = nondet['psfFluxErr'].values
            v     = sigma > 0
            lim   = np.full(sigma.shape, np.nan)
            lim[v] = -2.5 * np.log10(3 * sigma[v]) + ZP_AB
            ax.scatter(
                nondet['date'].values, lim,
                marker='v', color=color, s=30,
                alpha=0.45, zorder=3, label='_nolegend_')

    ax.invert_yaxis()   # convention astronomique : magnitudes croissantes vers le bas
    ax.set_ylabel("Magnitude AB")
    ax.legend(
        ncol=len(bands), loc='upper left',
        framealpha=0.85, handlelength=1.2,
        handletextpad=0.4, columnspacing=0.8)
    _apply_date_axis(ax, show_xlabel=True)


def make_lc_figure(oid, lc):
    """
    Crée et retourne une figure complète (flux + mag) pour un objet.
    Panneau flux en haut, panneau magnitude en bas, axe X partagé.
    """
    # Infos de la dernière alerte pour le titre
    lc_sorted = lc.dropna(subset=['midpointMjdTai']).sort_values(
        'midpointMjdTai', ascending=False)

    if not lc_sorted.empty:
        last       = lc_sorted.iloc[0]
        last_src   = last.get('diaSourceId', 'N/A')
        last_band  = last.get('band', 'N/A')
        last_date  = mjd_to_datestr(last.get('midpointMjdTai'))
        n_src      = len(lc)
        bands_used = ', '.join(_bands_in_order(lc)) or 'N/A'
    else:
        last_src = last_band = last_date = 'N/A'
        n_src = 0
        bands_used = 'N/A'

    main_title = (
        f"diaObjectId : {oid}   |   TAG : {TAG}\n"
        f"{n_src} sources   bandes : {bands_used}   "
        f"dernière alerte : {last_date}   filtre : {last_band}"
    )

    fig, (ax_flux, ax_mag) = plt.subplots(
        2, 1,
        figsize=(13, 8),
        sharex=True,
        layout='constrained',
        gridspec_kw={'height_ratios': [1, 1]}
    )
    fig.suptitle(main_title, fontsize=12, fontweight='bold')

    plot_lc_flux(ax_flux, lc)
    plot_lc_mag(ax_mag,  lc)

    # Supprimer les tick labels du panneau du haut (axe X partagé)
    plt.setp(ax_flux.xaxis.get_majorticklabels(), visible=False)

    return fig


print("Fonctions de tracé OK ✓")

## 6. Récupération des données

In [ ]:
from tqdm.notebook import tqdm
# ── Alertes ──
print(f"Récupération de {N_ALERTS} alertes '{TAG}'...")
df_tags = fetch_tag_alerts(TAG, N_ALERTS, STARTDATE, STOPDATE)
print(f"✓ {len(df_tags)} alertes reçues")

# ── Filtrage DDF (si activé) ──────────────────────────────────────────────────
if DDF_FILTER:
    print(f"\nFiltrage spatial DDF sur {len(df_tags)} alertes...")
    df_tags_ddf = filter_alerts_in_ddfs(df_tags, DDF_FIELDS)
    n_total  = len(df_tags)
    n_kept   = len(df_tags_ddf)
    n_reject = n_total - n_kept

     
    if df_tags_ddf.empty:
        print(f"⚠️  Aucune alerte dans les DDFs configurés.")
        print(f"   Vérifiez TAG='{TAG}', les dates ou élargissez DDF_FIELDS['radius_deg'].")
    else:
        print(f"\n✅ {n_kept} / {n_total} alertes retenues dans les DDFs "
              f"({n_reject} rejetées).")
        print(df_tags_ddf.groupby("ddf_name").size().rename("nb alertes").to_string())
        plot_ddf_map(df_tags_ddf, DDF_FIELDS)

    # Remplacement du DataFrame pour la suite du notebook
    id_col     = [c for c in df_tags_ddf.columns if "diaObjectId" in c][0]
    object_ids = df_tags_ddf[id_col].astype(str).unique().tolist()
    print(f"\n{len(object_ids)} objet(s) unique(s) après filtrage DDF.\n")

    df_tags = df_tags_ddf
else:
    print("DDF_FILTER=False — toutes les alertes conservées.")
    id_col = [c for c in df_tags.columns if 'diaObjectId' in c][0]
    object_ids = df_tags[id_col].astype(str).unique().tolist()
    print(f"Colonne ID : '{id_col}' — {len(object_ids)} objet(s) uniques\n")

ipy_display("df_tags_ddf: ",df_tags[['r:diaObjectId','r:ra','r:dec','r:band','r:diaSourceId']])


# ── Courbes de lumière ──
lightcurves = {}
for oid in tqdm(object_ids, desc='Téléchargement courbes de lumière', unit='obj'):
    try:
        lc = fetch_lightcurve(oid)
        ipy_display(f"lc {oid} (1):",lc)
        if lc.empty:
            print(f"  ⚠️  {oid} — aucune donnée")
        else:
            lightcurves[oid] = lc
            bands = sorted(lc['band'].dropna().unique()) if 'band' in lc.columns else []
            #ipy_display("lc (2):",lc)
            print(f"  ✓  {oid} — {len(lc)} sources, bandes : {bands}")
    except Exception as e:
        print(f"  ✗  {oid} — erreur : {e}")

print(f"\n{len(lightcurves)} objet(s) chargé(s) avec succès.")

## 7. Courbes de lumière — toutes bandes `ugrizy` superposées

Une figure par objet : **flux PSF (nJy)** en haut · **magnitude AB** en bas.  
Les flèches ▼ dans le panneau magnitude indiquent les non-détections (limite 3σ).

In [ ]:
for oid, lc in lightcurves.items():

    fig = make_lc_figure(oid, lc)

    # ── Sauvegarde ────────────────────────────────────────────────────────────
    if SAVE_FIGURES:
        stem = f"{OUTPUT_DIR}/lc_{oid}"
        fig.savefig(f"{stem}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{stem}.png", bbox_inches='tight', dpi=150)

    plt.show()

    # ── Lien cliquable vers le portail Fink/LSST ──────────────────────────────
    fink_url = f"https://lsst.fink-portal.org/{oid}"
    ipy_display(HTML(
        f'<div style="font-size:12px; margin:2px 0 16px 4px;">'
        f'🔗 <b>Portail Fink/LSST</b> — '
        f'diaObjectId <code>{oid}</code> : '
        f'<a href="{fink_url}" target="_blank">{fink_url}</a>'
        f'</div>'
    ))
    print()

print("\n✅ Toutes les courbes de lumière ont été générées.")

## 8. (Optionnel) Tableau récapitulatif

In [ ]:
rows = []
for oid, lc in lightcurves.items():
    lc_s = lc.dropna(subset=['midpointMjdTai']).sort_values(
        'midpointMjdTai', ascending=False)
    last = lc_s.iloc[0] if not lc_s.empty else {}
    rows.append({
        'diaObjectId'     : oid,
        'n_sources'       : len(lc),
        'bandes'          : ', '.join(_bands_in_order(lc)),
        'MJD_début'       : f"{lc['midpointMjdTai'].min():.4f}",
        'MJD_fin'         : f"{lc['midpointMjdTai'].max():.4f}",
        'flux_max (nJy)'  : f"{lc['psfFlux'].max():.1f}",
        'dernière_date'   : mjd_to_datestr(last.get('midpointMjdTai')) if 'midpointMjdTai' in last else '—',
        'dernier_filtre'  : last.get('band', '—'),
    })

df_summary = pd.DataFrame(rows)

df_summary['Portail Fink'] = df_summary['diaObjectId'].apply(
    lambda oid: f'<a href="https://lsst.fink-portal.org/{oid}" target="_blank">'
                f'lsst.fink-portal.org/{oid}</a>'
)

html = df_summary.to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse: collapse; font-size: 12px; width: 100%; }
  .fink-table th { background: #f0f4f8; padding: 6px 10px;
                   border-bottom: 2px solid #bbb; text-align: left; }
  .fink-table td { padding: 4px 10px; border-bottom: 1px solid #eee;
                   text-align: left; white-space: nowrap; }
  .fink-table tr:hover td { background: #f8f8ff; }
</style>
"""
ipy_display(HTML(style + html))